# 01 — Architecture and exact parameter budget

Instantiate the full configured model, count every trainable parameter (including all modality adapters/decoders), and confirm that the language-model head shares the same embedding matrix. This is a shape/budget validation, not training.

In [ ]:
import sys
from pathlib import Path

from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "phone-research.yaml").is_file() and (candidate / "src" / "onecore").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate OneCore project root from {current}")

root = find_project_root()
print("OneCore project root:", root)

sys.path.insert(0, str(root / "src"))

from onecore.config import load_config
from onecore.model import OneCore

cfg, _, _ = load_config(root / "configs" / "phone-research.yaml")
model = OneCore(cfg)
report = model.parameter_report()
assert model.lm_head.weight is model.token_embedding.weight
assert report["parameters"] == sum(p.numel() for p in model.parameters() if p.requires_grad)
assert 50_000_000 <= report["parameters"] <= 70_000_000, (
    f"Initial architecture is outside the agreed 50–70M budget: {report['parameters']:,}"
)
print(report)
print("One shared recurrent core; all trainable input/output codecs are included in this exact parameter count.")
